In [ ]:
from datetime import date, datetime

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import shap
from optuna import Trial, create_study, trial
from sklearn.metrics import mean_absolute_error, mean_squared_error

%matplotlib inline

# from optuna import Trial, trial, create_study

# from sklearn.preprocessing import OneHotEncoder

1.データ読み込み

In [ ]:
# 使用するカラムを指定

usecols = ["Year", "Month", "Day", "Hour", "Minute", "Temperature", "Clearsky GHI", "Cloud Type", "Dew Point", "DHI", "DNI",
       "Fill Flag", "GHI", "Relative Humidity", "Solar Zenith Angle","Surface Albedo", "Pressure", "Precipitable Water", "Wind Direction",
       "Wind Speed"]

In [ ]:
# ファイルの読み込み＋複数年を合算


df_weather_yamanashi = pd.DataFrame()

for year in range(2016, 2021):

    file_path = f"../data/weather_yamanashi_{year}.csv"
    print(file_path)
    df = pd.read_csv(file_path, skiprows=2, usecols=usecols)

    if df_weather_yamanashi.empty:
        # print("empty")
        df_weather_yamanashi = df
    else:
        # print("not empty")
        df_weather_yamanashi =  pd.concat([df_weather_yamanashi, df])

df_weather_yamanashi.reset_index(drop = True)

In [ ]:
# 前処理用のdfを作成
df_weather_yamanashi_preprocessing = df_weather_yamanashi.copy()

2.1モデルの変更

In [ ]:
# 非線形をとらえるために、線形回帰からlightgbmにモデルを変更

In [ ]:
# 時刻をつくる
# 元データがtz = 0。
# df_weather_yamanashi_preprocessing["utc"] = pd.to_datetime(df_weather_yamanashi_preprocessing[["Year", "Month", "Day", "Hour"]], utc = True)

df_weather_yamanashi_preprocessing["utc"] = pd.to_datetime(df_weather_yamanashi_preprocessing[["Year", "Month", "Day", "Hour"]])


In [ ]:
# time zoneの処理
# 元データはtz=0, local tz=9なので、hourを+9する。
# 23時とかは32時間とかになるからちゃんと計算する。

df_weather_yamanashi_preprocessing["jst"] = df_weather_yamanashi_preprocessing["utc"] + pd.Timedelta(hours=9)

In [ ]:
# 時刻関係を作成
df_weather_yamanashi_preprocessing["month_jst"] = df_weather_yamanashi_preprocessing["jst"].dt.month
df_weather_yamanashi_preprocessing["day_jst"] = df_weather_yamanashi_preprocessing["jst"].dt.day
df_weather_yamanashi_preprocessing["hour_jst"] = df_weather_yamanashi_preprocessing["jst"].dt.hour

In [ ]:
#日射量から、発電量を作成
# 発電量 = GHI[W/㎡] ＊ 1 [hour] * 発電効率　＊　温度補正 * パネル面積
# 発電量[Wh/m2] = GHI ＊ 1 ＊ 0.18 * (1 - 0.004 * (Temperature - 25)) * 1
# 本来は外気気温ではなく、パネル温度で補正をかける

# 発電量[kWh/m2] = GHI ＊ 1 ＊ 0.18 * (1 - 0.004 * (Temperature - 25)) * 1/1000

# [Wh]
df_weather_yamanashi_preprocessing["pv"] = df_weather_yamanashi_preprocessing["GHI"] * 1 * 0.18 * (1 - 0.004 * (df_weather_yamanashi_preprocessing["Temperature"] - 25))  * 1

In [ ]:
# データセット作成

def prepare_data(df_preprocessed, x_cols, y_col):
    
    feature_cols = x_cols + [y_col]

    train_start = datetime(2016, 1, 1, 0, 0, 0)
    # train_end = datetime(2018, 12, 31, 23, 59, 59)

    val_start = datetime(2019, 1, 1, 0, 0, 0)
    # val_end = datetime(2019, 12, 31, 23, 59, 59)

    test_start = datetime(2020, 1, 1, 0, 0, 0)
    # test_end = datetime(2020, 12, 31, 23, 59, 59)

    df_train = df_preprocessed[feature_cols][(df_preprocessed["jst"] >= train_start ) 
                                            & (df_preprocessed["jst"] < val_start)]
    df_val = df_preprocessed[feature_cols][(df_preprocessed["jst"] >= val_start ) 
                                            & (df_preprocessed["jst"] < test_start)]

    df_test =  df_preprocessed[feature_cols][df_preprocessed["jst"] >= test_start]

    # target = "pv"

    X_train = df_train[x_cols]
    y_train =df_train[y_col]
    y_train.name = y_col

    X_val = df_val[x_cols]
    y_val = df_val[y_col]
    y_val.name = y_col

    X_test = df_test[x_cols]
    y_test = df_test[y_col]
    y_test.name = y_col

    return X_train, y_train, X_val, y_val, X_test, y_test

In [ ]:
# 線形回帰と同じ特徴量でlightgbmで予測
y_col = "pv"
x_cols  = ["GHI", "month_jst", "day_jst", "hour_jst", "Temperature"]

In [ ]:
X_train, Y_train, X_val, Y_val, X_test, Y_test = prepare_data(df_weather_yamanashi_preprocessing, x_cols, y_col)

In [ ]:
# 学習、予測、評価を行う

def train_and_eval(X_train, Y_train, X_val, Y_val, X_test, Y_test):
    seed = 42

    model = lgb.LGBMRegressor(
        random_state = 42,
    )

    eval_set = [X_val.reset_index(drop=True), Y_val.reset_index(drop=True)]

    model.fit(
        X_train, 
        Y_train,
        eval_set = [(X_val, Y_val)],
        # verbose=-1 # 学習ログを省略
    )

    print("train done")

    Y_pred_test = model.predict(X_test)

    print("pred done")

    test_mae = mean_absolute_error(Y_test, Y_pred_test)
    test_rmse = mean_squared_error(Y_test, Y_pred_test)
    # test_nrmse = test_rmse / (np.max(Y_test)- np.min(Y_test))

    # print(f"mae:{test_mae}\nrmse:{test_rmse}\ntest_nrmse:{test_nrmse}\n")
    print(f"mae:{test_mae}\nrmse:{test_rmse}")
    
    df_test_pred = pd.DataFrame()
    df_test_pred["pv"] = Y_test
    df_test_pred["pred"] = Y_pred_test

    return test_mae, test_rmse, pd.concat([X_test, df_test_pred], axis=1), model
    

In [ ]:
# train_and_eval(X_train, Y_train, X_val, Y_val, X_test, Y_test)
mae_base, rmse_base, df_pred_base, model_base = train_and_eval(X_train, Y_train, X_val, Y_val, X_test, Y_test)


In [ ]:
# 誤差を作成
df_pred_base["residual"] = df_pred_base["pv"] - df_pred_base["pred"]

# df_pred_base[["hour_jst","residual"]].groupby("hour_jst").mean()

# 時間ごとに誤差の平均をプロット
plt.plot(df_pred_base[["hour_jst", "residual"]].groupby("hour_jst").mean())
plt.xlabel("hour_jst")
plt.ylabel("residual")
plt.show()

In [ ]:
# 時間ごとに誤差のばらつきが大きい
# sincosで周期化してみる

2.特徴量の調査

2.1-1時系列特徴量(周期性)

In [ ]:
df_sincos = df_weather_yamanashi_preprocessing.copy()

In [ ]:
# 年月日だと周期性を考慮できない可能性があり、day_of_year
# day of yearを作成
df_sincos["day_of_year"] = df_sincos["jst"].dt.dayofyear

In [ ]:
# うるう年の処理を含めて、day of yearをsin/cosにする

is_leap = df_sincos["jst"].dt.is_leap_year

max_days = np.where(is_leap, 366, 365)

normalized_day_of_year = (df_sincos["day_of_year"] - 1) / max_days

 # sin/cos変換
df_sincos["day_of_year_sin"] = np.sin(2 * np.pi * normalized_day_of_year.reset_index(drop=True))
df_sincos["day_of_year_cos"] = np.cos(2 * np.pi * normalized_day_of_year.reset_index(drop=True))

In [ ]:
# sincos化関数
def sin_cos(sr, duration, prefix):

    sin = np.sin(2 * np.pi * sr / duration)
    cos = np.cos(2 * np.pi * sr / duration)

    output = pd.DataFrame({
        f"{prefix}_sin": sin,
        f"{prefix}_cos": cos
    })

    return output

In [ ]:
df_sincos.head()

In [ ]:
# hourのsin cos化
df = pd.DataFrame()
df = sin_cos(df_sincos["hour_jst"], 24, "hour_jst")
df_sincos = pd.concat([df_sincos, df], axis=1)

In [ ]:
# weekdayをもとめる
# weekdayは大きさの序列が重要なのではなくて、何曜日かが重要なので、one-hot encodingする
# でも発電量にはそもそも曜日は関係ない気がする

# df_weather_yamanashi_preprocessing["weekday"] = df_weather_yamanashi_preprocessing["jst"].dt.dayofweek
# df_weather_yamanashi_preprocessing = pd.get_dummies(df_weather_yamanashi_preprocessing, columns=["weekday"])

# TODO
# 一旦特徴量には入れない
# 需要予測じゃないから、weekdayいらないかも

In [ ]:
# Minitueは全部０だからdropする？→dropじゃなくて、つかわない。
# ノイズになりそう

In [ ]:
# df_weather_yamanashi_preprocessing.drop("weekday", axis=1, inplace=True)

2.1-2 周期性特徴量の精度確認

In [ ]:
df_sincos.columns

In [ ]:
y_col = "pv"
x_cols  = ["GHI", "Temperature", "day_of_year_sin", "day_of_year_cos", "hour_jst_sin", "hour_jst_cos"]

In [ ]:
X_train, Y_train, X_val, Y_val, X_test, Y_test = prepare_data(df_sincos, x_cols, y_col)

In [ ]:
mae_sincos, rmse_sincos, df_pred_sincos, model_sincos = train_and_eval(X_train, Y_train, X_val, Y_val, X_test, Y_test)

In [ ]:
# 簡易的に時刻を戻す
# df_sincos_pred.reset_index(drop = True)
df_pred_sincos["recoverd_hour"] = np.arctan2(df_pred_sincos["hour_jst_sin"], df_pred_sincos["hour_jst_cos"])
df_pred_sincos["recoverd_hour"] = (df_pred_sincos["recoverd_hour"] + 2*np.pi) % (2*np.pi)
df_pred_sincos["recoverd_hour"] = df_pred_sincos["recoverd_hour"] * (24 / (2*np.pi))

In [ ]:
# 誤差を作成
df_pred_sincos["residual"] = df_pred_sincos["pv"] - df_pred_sincos["pred"]

In [ ]:
# df_sincos_pred[["hour","residual"]].groupby("hour").mean()

In [ ]:
# 時間ごとに誤差の平均をプロット
plt.plot(df_pred_sincos[["recoverd_hour","residual"]].groupby("recoverd_hour").mean())
plt.xlabel("hour")
plt.ylabel("residual")
plt.show()

In [ ]:
#　日射量の変化量が大きい朝方の誤差が大きい
# ラグや移動平均を入れてみる？

In [ ]:
# 気温ごとに誤差の平均をプロット
# plt.plot(df_sincos_pred[["Temperature","residual"]].groupby("Temperature").mean())
# plt.xlabel("Temperature")
# plt.ylabel("residual")
# plt.show()

2.2-1移動平均による平滑化

In [ ]:
df_lag = df_sincos.copy()

In [ ]:
# GHIのラグ作成
df_lag["GHI_rag1"] = df_lag["GHI"].shift(1)

# df_weather_yamanashi_preprocessing["GHI_rag2"] = df_weather_yamanashi_preprocessing["GHI"].shift(2)

# df_weather_yamanashi_preprocessing["GHI_rag3"] = df_weather_yamanashi_preprocessing["GHI"].shift(3)
 
# df_weather_yamanashi_preprocessing["GHI_rag24"]= df_weather_yamanashi_preprocessing["GHI"].shift(24)

# まずはrag1だけでやってみる

In [ ]:
# GHIの直近時間との差分
df_lag["delta_GHI"] = df_lag["GHI"]- df_lag["GHI_rag1"]

In [ ]:
#気温のラグ作成
df_lag["tempareture_rag1"] = df_lag["Temperature"].shift(1)
# .head(10)

In [ ]:
#気温の直近時間との差分
df_lag["delta_temperature"] = df_lag["Temperature"]- df_lag["tempareture_rag1"]

In [ ]:
#　GHIの移動平均
df_lag["GHI_ma"] = df_lag["GHI"].rolling(3).mean()

In [ ]:
#　temparatureの移動平均
df_lag["temparature_ma"] = df_lag["Temperature"].rolling(3).mean()

In [ ]:
df_lag.head()

2.2-2移動平均特徴量の精度確認

In [ ]:
df_lag.columns

In [ ]:
y_col = "pv"

x_cols = ["GHI", "Temperature", "day_of_year_sin", "day_of_year_cos", "hour_jst_sin", "hour_jst_cos", 
          "GHI_rag1", "delta_GHI", "tempareture_rag1", "delta_temperature", "GHI_ma", "temparature_ma"]

# x_cols = ["GHI", "Temperature", "day_of_year_sin", "day_of_year_cos", "hour_sin","hour_cos","GHI_rag1", "delta_GHI", "tempareture_rag1",
#        "delta_temperature", "GHI_ma", "temparature_ma"]


In [ ]:
X_train, Y_train, X_val, Y_val, X_test, Y_test = prepare_data(df_lag, x_cols, y_col)

In [ ]:
# train_and_eval(X_train, Y_train, X_val, Y_val, X_test, Y_test)
mae_lag, rmse_lag, df_pred_lag, model_lag = train_and_eval(X_train, Y_train, X_val, Y_val, X_test, Y_test)

In [ ]:
# 簡易的に時刻を戻す
# df_sincos_pred.reset_index(drop = True)
df_pred_lag["restored_hour"] = np.arctan2(df_pred_lag["hour_jst_sin"], df_pred_lag["hour_jst_cos"])
df_pred_lag["restored_hour"] = (df_pred_lag["restored_hour"] + 2*np.pi) % (2*np.pi)
df_pred_lag["restored_hour"] = df_pred_lag["restored_hour"] * (24 / (2*np.pi))

In [ ]:
# 誤差を作成
df_pred_lag["residual"] = df_pred_lag["pv"] - df_pred_lag["pred"]

In [ ]:
df_pred_lag[["hour","residual"]].groupby("hour").mean()

In [ ]:
# 時間ごとに誤差の平均をプロット
plt.plot(df_pred_lag[["restored_hour","residual"]].groupby("restored_hour").mean())
plt.xlabel("hour")
plt.ylabel("residual")
plt.show()

In [ ]:
# maeも下がってる上に、ばたつきの形がかわっただけ？

In [ ]:
# cloudtypeなどの外的要因ごとの精度確認
# 重複を避ける場合はdrop
df_merged = pd.merge(df_pred_lag, df_weather_yamanashi_preprocessing, left_index = True, right_index = True, how = "inner")

In [ ]:
df_merged.columns

In [ ]:
# cloudtypeごとに誤差の平均をプロット
plt.plot(df_merged[["Cloud Type","residual"]].groupby("Cloud Type").mean())
plt.xlabel("Cloud Type")
plt.ylabel("residual")
plt.show()

In [ ]:
# cloudtypeごとに誤差のばらつきが大きい
# モデルが拾えてない可能性？

In [ ]:
# Dew Pointごとに誤差の平均をプロット
plt.plot(df_merged[["Dew Point","residual"]].groupby("Dew Point").mean())
plt.xlabel("Cloud Dew")
plt.ylabel("residual")
plt.show()

In [ ]:
# 特定の値付近で誤差が大きい

In [ ]:
# Relative Humidityごとに誤差の平均をプロット
plt.plot(df_merged[["Relative Humidity","residual"]].groupby("Relative Humidity").mean())
plt.xlabel("Relative Humidity")
plt.ylabel("residual")
plt.show()

In [ ]:
# 湿度が低いほど誤差が大きい？

3.1-1外的特徴量

In [ ]:
# 晴天度の指標
#　GHIの理論値とGHIの比率
# df_weather_yamanashi_preprocessing["clear_sky_index"] = df_weather_yamanashi_preprocessing["GHI"] / df_weather_yamanashi_preprocessing["Clearsky GHI"]

# TODO
# 一旦保留
#夜などはclearsky GHIが0になる。
# そのため、０割になってnanができる
# 原因がそもそも日射量が出ていないことでの０割なので、nanを0にする？それか、nanがある
# めっちゃ曇りでGHIが０ってありえる？曇りでも雨でも多少はGHIありそう
# というか、理論値と実際の値の比較って、リークにならない？clearGHIの定義の確認。あっいいのか。雲がないのがclearGHIで、予測がGHIなら。
# そうすると、cloudtypeとだぶる？

In [ ]:
df_added_weather = df_lag.copy()

In [ ]:
# wind directionのsin cos化

# 関数結果をいれるための一時的な格納用でdfを作成
df = pd.DataFrame()

# データ品質的に問題があれば、つまり、0~360の値に収まっていない場合は、念の為360でmodをとる
df = sin_cos(df_added_weather["Wind Direction"], 360, "wind_direction")

df_added_weather = pd.concat([df_added_weather, df], axis=1)


In [ ]:
# 質的変数であるCloudTypeをone-hot encoding>ラベルエンコーディングのまま、dfの列の方をcategoryにする

df_added_weather["Cloud Type"] = df_added_weather["Cloud Type"].astype("category")


In [ ]:
df_weather_yamanashi_preprocessing.columns

3.1-2外的特徴量の精度確認

In [ ]:
y_col = "pv"

#気象データ９個
x_cols = ["GHI", "Temperature", "day_of_year_sin", "day_of_year_cos", "hour_jst_sin","hour_jst_cos","Cloud Type","Dew Point", "Relative Humidity", "Solar Zenith Angle", "Surface Albedo", "Pressure",
       "Precipitable Water", "Wind Direction", "Wind Speed"]

#cloud typeのみ
# x_cols = ["GHI", "Temperature", "day_of_year_sin", "day_of_year_cos", "hour_jst_sin","hour_jst_cos","Cloud Type"]

In [ ]:
X_train, Y_train, X_val, Y_val, X_test, Y_test = prepare_data(df_added_weather, x_cols, y_col)


In [ ]:
mae_weather, rmse_weather, df_pred_weather, model_weather = train_and_eval(X_train, Y_train, X_val, Y_val, X_test, Y_test)


3.1-3feature importanceの確認

In [ ]:
# gain = model.feature_importance(importance_type='gain')
# feature_names = model.feature_name_

In [ ]:
lgb.plot_importance(model, importance_type="gain")

In [ ]:
importance_gain = model.booster_.feature_importance(importance_type="gain")

df_importance = pd.DataFrame({
    "feature": model.feature_name_,
    "importance": importance_gain
}).sort_values("importance", ascending=False)

In [ ]:
df_importance

In [ ]:
# shapの計算
shap_explainer = shap.TreeExplainer(model)

In [ ]:
shap_values = shap_explainer.shap_values(X_test)

In [ ]:
shap.summary_plot(shap_values, X_test)

4.GHIの有無による精度比較

4.1-1GHIあり

In [ ]:
y_col = "pv"
x_cols = ["GHI", "Temperature", "day_of_year", "day_of_year_sin", "day_of_year_cos", "hour_jst_sin","hour_jst_cos","Cloud Type","Dew Point", "Relative Humidity", "Solar Zenith Angle", "Surface Albedo", "Pressure",
       "Precipitable Water", "Wind Direction", "Wind Speed"]

In [ ]:
X_train, Y_train, X_val, Y_val, X_test, Y_test = prepare_data(df_added_weather, x_cols, y_col)

In [ ]:
mae_with_ghi, rmse_with_ghi, df_pred_with_ghi, model_with_ghi = train_and_eval(X_train, Y_train, X_val, Y_val, X_test, Y_test)

4.1-2GHIなし

In [ ]:
y_col = "pv"
# x_cols = ["Temperature", "day_of_year", "day_of_year_sin", "day_of_year_cos", "hour_sin","hour_cos"]

#気象データ９個
x_cols = ["Temperature", "day_of_year", "day_of_year_sin", "day_of_year_cos", "hour_jst_sin","hour_jst_cos","Cloud Type","Dew Point", "Relative Humidity", "Solar Zenith Angle", "Surface Albedo", "Pressure",
       "Precipitable Water", "Wind Direction", "Wind Speed"]

In [ ]:
X_train, Y_train, X_val, Y_val, X_test, Y_test = prepare_data(df_added_weather, x_cols, y_col)

In [ ]:
mae_no_ghi, rmse_no_ghi, df_pred_no_ghi, model_no_ghi = train_and_eval(X_train, Y_train, X_val, Y_val, X_test, Y_test)

In [ ]:
importance_gain = model.booster_.feature_importance(importance_type="gain")

df_importance = pd.DataFrame({
    "feature": model.feature_name_,
    "importance": importance_gain
}).sort_values("importance", ascending=False)

In [ ]:
df_importance

In [ ]:
# shapの計算.GHIあり。
shap_explainer_with_ghi = shap.TreeExplainer(model_with_ghi)
shap_values_with_ghi = shap_explainer_with_ghi.shap_values(X_test)
shap.summary_plot(shap_values_with_ghi, X_test)

In [ ]:
# shapの計算.GHIなし。
shap_explainer_no_ghi = shap.TreeExplainer(model_no_ghi)
shap_values_no_ghi = shap_explainer_no_ghi.shap_values(X_test)
shap.summary_plot(shap_values_no_ghi, X_test)

5.1optunaによるパラメータ調整

In [ ]:
y_col = "pv"
x_cols = ["GHI", "Temperature", "day_of_year_sin", "day_of_year_cos", "hour_jst_sin", "hour_jst_cos"]

In [ ]:
X_train, Y_train, X_val, Y_val, X_test, Y_test = prepare_data(df_lag, x_cols, y_col)

In [ ]:
def objective(trial, X_train, Y_train, X_val, Y_val, X_test, Y_test):
    # 探索するパラメータの範囲を指定
    params = {
        "num_leaves": trial.suggest_int("num_leaves", 16, 128),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 20, 200),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),

        "lambda_l1": trial.suggest_float("lambda_l1", 1e-3, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-3, 10.0, log=True),
        "random_state": 42,
        "verbosity": -1 # ログを非表示にする
    }

    callbacks = [
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=0) # ログを非表示にする場合
    ]

    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_train, 
        Y_train,
        eval_set = [(X_val, Y_val)],
        eval_metric = "mae",
        callbacks = callbacks,
        
    )
    
    preds = model.predict(X_test)
    mae = mean_absolute_error(Y_test, preds)

    return mae

In [ ]:
study = optuna.create_study(direction = "minimize")

In [ ]:
study.optimize(
    lambda trial: objective(trial, X_train, Y_train, X_val, Y_val, X_test, Y_test), 
    n_trials=100
)

In [ ]:
print(f"Best value: {study.best_value}")
print(f"Best params: {study.best_params}")

5.2best parameterでモデル構築

In [ ]:
best_params = study.best_params

In [ ]:
model_best_params = lgb.LGBMRegressor(
    **best_params,
    random_state = 42,
)

eval_set = [X_val.reset_index(drop=True), Y_val.reset_index(drop=True)]

model_best_params.fit(
    X_train, 
    Y_train,
    eval_set = eval_set,
    # verbose=-1 # 学習ログを省略
)

print("train done")

Y_pred_test = model_best_params.predict(X_test)

print("pred done")

test_mae = mean_absolute_error(Y_test, Y_pred_test)
test_rmse = mean_squared_error(Y_test, Y_pred_test)
# test_nrmse = test_rmse / (np.max(Y_test)- np.min(Y_test))

# print(f"mae:{test_mae}\nrmse:{test_rmse}\ntest_nrmse:{test_nrmse}\n")
print(f"mae:{test_mae}\nrmse:{test_rmse}")

Appendix:ポートフォリオ用のグラフ作成

In [ ]:
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.figsize": (8, 4.5),
    "axes.edgecolor": "#333333",
    "axes.linewidth": 0.8,
})

In [ ]:
# 時間での誤差の分析

residual_by_hour = (
    df_pred_base[["hour_jst", "residual"]]
    .groupby("hour_jst")
    .mean()
    .reset_index()
)

plt.figure(figsize=(8, 4.5))

# 折れ線
plt.plot(
    residual_by_hour["hour_jst"],
    residual_by_hour["residual"],
    marker="o",
    linewidth=1.5
)

# ゼロライン
plt.axhline(0, linestyle="--", linewidth=1)

# ラベル
plt.xlabel("Hour")
plt.ylabel("Mean Residual")
plt.title("Residual by Hour (LightGBM baseline)")

# 目盛り
plt.xticks(range(0, 24, 2))

# グリッド
plt.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.savefig("../images/residual_baseline.png", dpi=300)
plt.show()

In [ ]:
# 時間での誤差の分析 sincos後
# plt.plot(df_pred_sincos[["recoverd_hour","residual"]].groupby("recoverd_hour").mean())
# plt.xlabel("recoverd_hour")
# plt.ylabel("residual")
# plt.show()

residual_by_hour = (
    df_pred_sincos[["recoverd_hour", "residual"]]
    .groupby("recoverd_hour")
    .mean()
    .reset_index()
)

plt.figure(figsize=(8, 4.5))

# 折れ線
plt.plot(
    residual_by_hour["recoverd_hour"],
    residual_by_hour["residual"],
    marker="o",
    linewidth=1.5
)

# ゼロライン
plt.axhline(0, linestyle="--", linewidth=1)

# ラベル
plt.xlabel("Hour")
plt.ylabel("Mean Residual")
plt.title("Residual by Hour (LightGBM baseline)")

# 目盛り
plt.xticks(range(0, 24, 2))

# グリッド
plt.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.savefig("../images/residual_sin_cos.png", dpi=300)
plt.show()

In [ ]:
# model_comparison.png

models = [
    "Linear",
    "LightGBM",
    "+ sin/cos",
    "+ Optuna"
]

mae = [1.184, 0.113, 0.113, 0.097]

colors = ["#B0B0B0", "#B0B0B0", "#B0B0B0", "#2E6FBB"]

plt.figure()
plt.bar(models, mae, color=colors)

plt.ylabel("MAE")
plt.title("Model Improvement")

plt.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.savefig("../images/model_comparison.png", dpi=300)
plt.show()

In [ ]:
# feature_comparison.png
features = [
    "sin/cos",
    # "+ CloudType",
    "+ Weather(9)",
    "+ Lag/MA"
]

mae = [0.113, 0.118, 0.119]
# mae = [0.1132, 0.1141, 0.1184, 0.1194]


plt.figure()
plt.barh(features, mae, color="#4A4A4A")

plt.xlabel("MAE")
plt.title("Feature Impact")

plt.grid(axis="x", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.savefig("../images/feature_comparison.png", dpi=300)
plt.show()

In [ ]:
#  ghi_vs_no_ghi.png
labels = ["With GHI", "Without GHI"]
mae = [0.118, 4.874]

colors = ["#2E6FBB", "#D9534F"]

plt.figure()
plt.bar(labels, mae, color=colors)

plt.ylabel("MAE")
plt.title("Impact of GHI")

plt.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.savefig("../images/ghi_vs_no_ghi.png", dpi=300)
plt.show()

In [ ]:
#  SHAP
# shapの計算.GHIあり。
shap_explainer_with_ghi = shap.TreeExplainer(model_with_ghi)
shap_values_with_ghi = shap_explainer_with_ghi.shap_values(X_test)

shap.summary_plot(shap_values_with_ghi, X_test, show=False)
plt.gcf().set_size_inches(8, 5)
plt.tight_layout()
plt.savefig("../images/shap_with_ghi.png", dpi=300)
plt.show()

In [ ]:
#  SHAP
# shapの計算.GHIなし。
shap_explainer_no_ghi = shap.TreeExplainer(model_no_ghi)
shap_values_no_ghi = shap_explainer_no_ghi.shap_values(X_test)

shap.summary_plot(shap_values_no_ghi, X_test, show=False)
plt.gcf().set_size_inches(8, 5)
plt.tight_layout()
plt.savefig("../images/shap_no_ghi.png", dpi=300)
plt.show()